In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

In [2]:
matches_df = pd.read_csv("../data/processed/matches.csv")
matches_df.head()

,datetime,league,year,match_id,home_team_id,home_team,home_short,away_team_id,away_team,away_short,home_goals,away_goals,home_xG,away_xG,total_goals,total_xG
0,2020-09-18 18:30:00,bundesliga,2020,14173,117,Bayern Munich,BAY,124,Schalke 04,SCH,8,0,4.620880,0.149750,8,4.770630
1,2020-09-19 13:30:00,bundesliga,2020,14174,132,Eintracht Frankfurt,EIN,262,Arminia Bielefeld,ARM,1,1,2.469280,0.618787,2,3.088067
2,2020-09-19 13:30:00,bundesliga,2020,14175,240,Union Berlin,UNI,121,Augsburg,AUG,1,3,1.045150,1.420460,4,2.465610
3,2020-09-19 13:30:00,bundesliga,2020,14176,134,FC Cologne,COL,120,Hoffenheim,HOF,2,3,2.408010,2.903200,5,5.311210
4,2020-09-19 13:30:00,bundesliga,2020,14177,123,Werder Bremen,WER,122,Hertha Berlin,HER,1,4,0.495892,2.054060,5,2.549952


In [3]:
teams_df = pd.read_csv("../data/processed/teams.csv")
teams_df.head()

,h_a,xG,xGA,npxG,npxGA,ppda,ppda_allowed,deep,deep_allowed,scored,...,date,wins,draws,loses,pts,npxGD,team_id,team_name,league,year
0,h,4.62088,0.149750,3.86311,0.149750,"{'att': 203, 'def': 32}","{'att': 380, 'def': 15}",24,3,8,...,2020-09-18 18:30:00,1,0,0,3,3.713360,117,Bayern Munich,bundesliga,2020
1,a,1.17311,2.523390,1.17311,1.765620,"{'att': 99, 'def': 17}","{'att': 341, 'def': 26}",11,6,1,...,2020-09-27 13:30:00,0,0,1,0,-0.592510,117,Bayern Munich,bundesliga,2020
2,h,3.99311,1.245090,3.23533,1.245090,"{'att': 157, 'def': 38}","{'att': 280, 'def': 23}",11,1,4,...,2020-10-04 16:00:00,1,0,0,3,1.990240,117,Bayern Munich,bundesliga,2020
3,a,3.29447,1.212350,3.29447,1.212350,"{'att': 223, 'def': 30}","{'att': 416, 'def': 12}",16,4,4,...,2020-10-17 16:30:00,1,0,0,3,2.082120,117,Bayern Munich,bundesliga,2020
4,h,3.80823,0.429229,3.80823,0.429229,"{'att': 220, 'def': 22}","{'att': 372, 'def': 22}",12,1,5,...,2020-10-24 13:30:00,1,0,0,3,3.379001,117,Bayern Munich,bundesliga,2020


In [4]:
home_df = (teams_df[teams_df['h_a']=='h'].groupby(['team_name','league','year'])
           [['scored','missed','xG','xGA']].sum().reset_index())

home_df.columns = ['team_name','league','year','scored_h','conceded_h','xG_h','xGA_h']

away_df = (teams_df[teams_df['h_a']=='a'].groupby(['team_name','league','year'])
           [['scored','missed','xG','xGA']].sum().reset_index())

away_df.columns = ['team_name','league','year','scored_a','conceded_a','xG_a','xGA_a']


In [5]:
seasonal_df = home_df.merge(away_df, on=['team_name','league','year'])

seasonal_df['scored'] = seasonal_df['scored_h'] + seasonal_df['scored_a']
seasonal_df['conceded'] = seasonal_df['conceded_h'] + seasonal_df['conceded_a']
seasonal_df['xG'] = seasonal_df['xG_h'] + seasonal_df['xG_a']
seasonal_df['xGA'] = seasonal_df['xGA_h'] + seasonal_df['xGA_a']

seasonal_df['xGD_Attack'] = seasonal_df['scored'] - seasonal_df['xG']
seasonal_df['xGD_Defence'] = seasonal_df['conceded'] - seasonal_df['xGA']

seasonal_df['xGD_Attack_h'] = seasonal_df['scored_h'] - seasonal_df['xG_h']
seasonal_df['xGD_Attack_a'] = seasonal_df['scored_a'] - seasonal_df['xG_a']
seasonal_df['xGD_Defence_h'] = seasonal_df['conceded_h'] - seasonal_df['xGA_h']
seasonal_df['xGD_Defence_a'] = seasonal_df['conceded_a'] - seasonal_df['xGA_a']

In [6]:
seasonal_df.head()

,team_name,league,year,scored_h,conceded_h,xG_h,xGA_h,scored_a,conceded_a,xG_a,...,scored,conceded,xG,xGA,xGD_Attack,xGD_Defence,xGD_Attack_h,xGD_Attack_a,xGD_Defence_h,xGD_Defence_a
0,AC Milan,serie_a,2020,31,24,37.928325,24.530668,43,17,37.119001,...,74,41,75.047326,48.838073,-1.047326,-7.838073,-6.928325,5.880999,-0.530668,-7.307405
1,AC Milan,serie_a,2021,28,12,32.436844,14.717525,41,19,34.901264,...,69,31,67.338108,34.631340,1.661892,-3.631340,-4.436844,6.098736,-2.717525,-0.913815
2,AC Milan,serie_a,2022,41,20,40.158294,17.496600,23,23,29.742495,...,64,43,69.900789,40.109474,-5.900789,2.890526,0.841706,-6.742495,2.503400,0.387126
3,AC Milan,serie_a,2023,38,17,37.496432,21.897890,38,32,34.359323,...,76,49,71.855755,47.731906,4.144245,1.268094,0.503568,3.640677,-4.897890,6.165984
4,AC Milan,serie_a,2024,30,16,31.668365,20.520671,31,27,37.841776,...,61,43,69.510141,47.271499,-8.510141,-4.271499,-1.668365,-6.841776,-4.520671,0.249172


In [7]:
seasonal_df[seasonal_df['team_name'] == 'Bayern Munich'][['scored','conceded','xG','xGA']]

,scored,conceded,xG,xGA
60,99,44,75.932410,38.873343
61,97,37,99.905525,38.609638
62,92,38,85.118184,37.006959
63,94,45,94.400968,35.224795
64,99,32,95.017114,27.987424


In [8]:
seasonal_df[(seasonal_df['team_name'] == 'Bayern Munich')]

,team_name,league,year,scored_h,conceded_h,xG_h,xGA_h,scored_a,conceded_a,xG_a,...,scored,conceded,xG,xGA,xGD_Attack,xGD_Defence,xGD_Attack_h,xGD_Attack_a,xGD_Defence_h,xGD_Defence_a
60,Bayern Munich,bundesliga,2020,64,21,46.027431,18.501977,35,23,29.904979,...,99,44,75.932410,38.873343,23.067590,5.126657,17.972569,5.095021,2.498023,2.628634
61,Bayern Munich,bundesliga,2021,48,15,49.282000,19.077494,49,22,50.623525,...,97,37,99.905525,38.609638,-2.905525,-1.609638,-1.282000,-1.623525,-4.077494,2.467856
62,Bayern Munich,bundesliga,2022,53,17,49.365890,16.183654,39,21,35.752294,...,92,38,85.118184,37.006959,6.881816,0.993041,3.634110,3.247706,0.816346,0.176695
63,Bayern Munich,bundesliga,2023,53,12,52.129240,13.131342,41,33,42.271728,...,94,45,94.400968,35.224795,-0.400968,9.775205,0.870760,-1.271728,-1.131342,10.906547
64,Bayern Munich,bundesliga,2024,53,16,55.768840,13.508277,46,16,39.248274,...,99,32,95.017114,27.987424,3.982886,4.012576,-2.768840,6.751726,2.491723,1.520853


In [9]:
matches_df

,datetime,league,year,match_id,home_team_id,home_team,home_short,away_team_id,away_team,away_short,home_goals,away_goals,home_xG,away_xG,total_goals,total_xG
0,2020-09-18 18:30:00,bundesliga,2020,14173,117,Bayern Munich,BAY,124,Schalke 04,SCH,8,0,4.620880,0.149750,8,4.770630
1,2020-09-19 13:30:00,bundesliga,2020,14174,132,Eintracht Frankfurt,EIN,262,Arminia Bielefeld,ARM,1,1,2.469280,0.618787,2,3.088067
2,2020-09-19 13:30:00,bundesliga,2020,14175,240,Union Berlin,UNI,121,Augsburg,AUG,1,3,1.045150,1.420460,4,2.465610
3,2020-09-19 13:30:00,bundesliga,2020,14176,134,FC Cologne,COL,120,Hoffenheim,HOF,2,3,2.408010,2.903200,5,5.311210
4,2020-09-19 13:30:00,bundesliga,2020,14177,123,Werder Bremen,WER,122,Hertha Berlin,HER,1,4,0.495892,2.054060,5,2.549952
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8977,2025-05-25 18:45:00,serie_a,2024,27733,107,Atalanta,ATA,230,Parma Calcio 1913,PAR,2,3,1.902650,2.825350,5,4.728000
8978,2025-05-25 18:45:00,serie_a,2024,27736,108,Empoli,EMP,94,Verona,VER,1,2,1.233510,0.833807,3,2.067317
8979,2025-05-25 18:45:00,serie_a,2024,27737,96,Lazio,LAZ,243,Lecce,LEC,0,1,1.735210,0.708667,1,2.443877
8980,2025-05-25 18:45:00,serie_a,2024,27740,113,Torino,TOR,95,Roma,ROM,0,2,0.381059,2.644360,2,3.025419


In [10]:
teams_df['date'] = pd.to_datetime(teams_df['date'])
matches_df['datetime'] = pd.to_datetime(matches_df['datetime'])

# merge home rows
home_df = teams_df[teams_df['h_a'] == 'h'].merge(
    matches_df[['datetime','league','year','match_id','home_team','away_team']],
    left_on=['date','league','year','team_name'],
    right_on=['datetime','league','year','home_team'],
    how='left'
)

# merge away rows
away_df = teams_df[teams_df['h_a'] == 'a'].merge(
    matches_df[['datetime','league','year','match_id','home_team','away_team']],
    left_on=['date','league','year','team_name'],
    right_on=['datetime','league','year','away_team'],
    how='left'
)

clinical_df = pd.concat([home_df, away_df], ignore_index=True)

# verify
print(clinical_df[(clinical_df['team_name'] == 'Manchester City') & (clinical_df['year'] == 2023)].shape[0])
print(clinical_df['match_id'].isna().sum())



38
0


In [11]:
clinical_df

,h_a,xG,xGA,npxG,npxGA,ppda,ppda_allowed,deep,deep_allowed,scored,...,pts,npxGD,team_id,team_name,league,year,datetime,match_id,home_team,away_team
0,h,4.62088,0.149750,3.86311,0.149750,"{'att': 203, 'def': 32}","{'att': 380, 'def': 15}",24,3,8,...,3,3.713360,117,Bayern Munich,bundesliga,2020,2020-09-18 18:30:00,14173,Bayern Munich,Schalke 04
1,h,3.99311,1.245090,3.23533,1.245090,"{'att': 157, 'def': 38}","{'att': 280, 'def': 23}",11,1,4,...,3,1.990240,117,Bayern Munich,bundesliga,2020,2020-10-04 16:00:00,15166,Bayern Munich,Hertha Berlin
2,h,3.80823,0.429229,3.80823,0.429229,"{'att': 220, 'def': 22}","{'att': 372, 'def': 22}",12,1,5,...,3,3.379001,117,Bayern Munich,bundesliga,2020,2020-10-24 13:30:00,15180,Bayern Munich,Eintracht Frankfurt
3,h,1.67360,1.631040,1.67360,1.631040,"{'att': 129, 'def': 33}","{'att': 266, 'def': 16}",18,4,1,...,1,0.042560,117,Bayern Munich,bundesliga,2020,2020-11-21 14:30:00,15203,Bayern Munich,Werder Bremen
4,h,1.00047,1.564920,1.00047,1.564920,"{'att': 237, 'def': 19}","{'att': 345, 'def': 15}",6,1,3,...,1,-0.564450,117,Bayern Munich,bundesliga,2020,2020-12-05 17:30:00,15221,Bayern Munich,RasenBallsport Leipzig
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17959,a,1.35523,1.728030,1.35523,1.728030,"{'att': 216, 'def': 26}","{'att': 242, 'def': 26}",15,5,1,...,0,-0.372800,286,Como,serie_a,2024,2025-03-15 17:00:00,27647,AC Milan,Como
17960,a,2.27029,1.059710,2.27029,1.059710,"{'att': 298, 'def': 26}","{'att': 371, 'def': 22}",7,1,3,...,3,1.210580,286,Como,serie_a,2024,2025-04-05 13:00:00,27663,Monza,Como
17961,a,1.65387,0.666737,1.65387,0.666737,"{'att': 132, 'def': 25}","{'att': 186, 'def': 20}",4,1,3,...,3,0.987133,286,Como,serie_a,2024,2025-04-19 13:00:00,27688,Lecce,Como
17962,a,1.32818,2.745250,1.32818,2.745250,"{'att': 193, 'def': 21}","{'att': 306, 'def': 27}",10,9,1,...,3,-1.417070,286,Como,serie_a,2024,2025-05-03 13:00:00,27709,Parma Calcio 1913,Como


In [12]:
# Finishing Attributes (per team)
clinical_df['match_xGD']     = clinical_df['scored'] - clinical_df['xG']
clinical_df['match_npxGD']   = clinical_df['scored'] - clinical_df['npxG']
clinical_df['xGA_diff']      = clinical_df['missed'] - clinical_df['xGA']
clinical_df['dominance']     = clinical_df['xG']     - clinical_df['xGA']

clinical_df['clinical']       = np.where((clinical_df['match_xGD'] > 0)     & (clinical_df['xG'] > 0.5), 'clinical',       'not_clinical')
clinical_df['ultra_clinical'] = np.where((clinical_df['match_xGD'] >= 1.5)  & (clinical_df['xG'] > 0.5), 'ultra_clinical',  'not_ultra_clinical')
clinical_df['wasteful']       = np.where((clinical_df['match_xGD'] < 0)     & (clinical_df['xG'] > 0.5), 'wasteful',        'not_wasteful')
clinical_df['ultra_wasteful'] = np.where((clinical_df['match_xGD'] <= -1.5) & (clinical_df['xG'] > 0.5), 'ultra_wasteful',  'not_ultra_wasteful')
clinical_df['expected']       = np.where((clinical_df['match_xGD'].abs() <= 0.1) & (clinical_df['xG'] > 0.5), 'expected',   'not_expected')

# Result vs Expectations
clinical_df['heist']          = np.where((clinical_df['result'] == 'w') & (clinical_df['xG'] < clinical_df['xGA']),            'heist',          'not_heist')
clinical_df['grand_heist']    = np.where((clinical_df['result'] == 'w') & (clinical_df['xGA'] - clinical_df['xG'] >= 1.0),     'grand_heist',    'not_grand_heist')
clinical_df['robbery']        = np.where((clinical_df['result'] == 'l') & (clinical_df['xG'] > clinical_df['xGA']),             'robbery',        'not_robbery')
clinical_df['grand_robbery']  = np.where((clinical_df['result'] == 'l') & (clinical_df['xG'] - clinical_df['xGA'] >= 1.0),     'grand_robbery',  'not_grand_robbery')
clinical_df['dropped_points'] = np.where((clinical_df['result'] == 'd') & (clinical_df['xG'] - clinical_df['xGA'] >= 1.0),     'dropped_points', 'not_dropped_points')
clinical_df['stolen_point']   = np.where((clinical_df['result'] == 'd') & (clinical_df['xGA'] - clinical_df['xG'] >= 1.0),     'stolen_point',   'not_stolen_point')
clinical_df['dominant_win']   = np.where((clinical_df['result'] == 'w') & (clinical_df['xG'] - clinical_df['xGA'] >= 1.5),     'dominant_win',   'not_dominant_win')

# Match Character — deduplicated to avoid double counting
match_level_df = clinical_df[clinical_df['h_a'] == 'h'].copy()
match_level_df['match_total_goals'] = match_level_df['scored'] + match_level_df['missed']
match_level_df['chaos']        = np.where(match_level_df['match_total_goals'] > (match_level_df['xG'] + match_level_df['xGA']) * 1.5, 'chaos',        'not_chaos')
match_level_df['ghost']        = np.where(match_level_df['match_total_goals'] < (match_level_df['xG'] + match_level_df['xGA']) * 0.5, 'ghost',        'not_ghost')
match_level_df['keepers_day']  = np.where((match_level_df['xG'] + match_level_df['xGA'] >= 3.0) & (match_level_df['match_total_goals'] <= 1), 'keepers_day',  'not_keepers_day')
match_level_df['paradise_day'] = np.where((match_level_df['xG'] + match_level_df['xGA'] <= 1.5) & (match_level_df['match_total_goals'] >= 3), 'paradise_day', 'not_paradise_day')

clinical_df = clinical_df.merge(
    match_level_df[['match_id','match_total_goals','chaos','ghost','keepers_day','paradise_day']],
    on='match_id', how='left'
)

# Value columns paired with flag columns
clinical_df['xGD_magnitude']          = clinical_df['match_xGD'].abs().round(3)
clinical_df['dominance_gap']          = (clinical_df['xGA'] - clinical_df['xG']).round(3)
clinical_df['goals_vs_xG_ratio']      = (clinical_df['match_total_goals'] / (clinical_df['xG'] + clinical_df['xGA']).clip(lower=0.01)).round(3)
clinical_df['goals_vs_xG_ratio_team'] = (clinical_df['scored'] / clinical_df['xG'].clip(lower=0.01)).round(3)
clinical_df['gk_save_ratio']          = (1 - (clinical_df['missed'] / clinical_df['xGA'].clip(lower=0.01))).round(3)
clinical_df['xG_total']               = (clinical_df['xG'] + clinical_df['xGA']).round(3)

# Defensive Character
clinical_df['heroic_defence']      = np.where((clinical_df['missed'] == 0) & (clinical_df['xGA'] >= 1.5),  'heroic_defence',      'not_heroic_defence')
clinical_df['routine_clean_sheet'] = np.where((clinical_df['missed'] == 0) & (clinical_df['xGA'] < 0.5),   'routine_clean_sheet', 'not_routine_clean_sheet')
clinical_df['defensive_collapse']  = np.where((clinical_df['missed'] >= 3) & (clinical_df['xGA'] < 1.5),   'defensive_collapse',  'not_defensive_collapse')
clinical_df['gk_worldie']          = np.where((clinical_df['xGA'] - clinical_df['missed']) >= 1.5,          'gk_worldie',          'not_gk_worldie')
clinical_df['gk_nightmare']        = np.where((clinical_df['missed'] - clinical_df['xGA']) >= 1.5,          'gk_nightmare',        'not_gk_nightmare')

# Momentum Based Character
clinical_df['away_win']     = np.where((clinical_df['result'] == 'w') & (clinical_df['h_a'] == 'a'),                                                'away_win',     'not_away_win')
clinical_df['away_heist']   = np.where((clinical_df['result'] == 'w') & (clinical_df['h_a'] == 'a') & (clinical_df['xG'] < clinical_df['xGA']),     'away_heist',   'not_away_heist')
clinical_df['home_bottled'] = np.where((clinical_df['result'] != 'w') & (clinical_df['h_a'] == 'h') & (clinical_df['xG'] > clinical_df['xGA']),     'home_bottled', 'not_home_bottled')

# Full Story
clinical_df['perfect_heist'] = np.where((clinical_df['heist'] == 'heist') & (clinical_df['h_a'] == 'a') & (clinical_df['xGA'] - clinical_df['xG'] >= 1.0), 'perfect_heist', 'not_perfect_heist')
clinical_df['cruel']         = np.where((clinical_df['grand_robbery'] == 'grand_robbery') & (clinical_df['h_a'] == 'a'),                                    'cruel',         'not_cruel')
clinical_df['fortress']      = np.where((clinical_df['heist'] == 'heist') & (clinical_df['h_a'] == 'h'),                                                    'fortress',      'not_fortress')

# Game Tag — always last
clinical_df['game_tag'] = 'Normal'
for col, val, tag in reversed([
    ('perfect_heist', 'perfect_heist', 'Perfect Heist'),
    ('grand_heist',   'grand_heist',   'Grand Heist'),
    ('grand_robbery', 'grand_robbery', 'Grand Robbery'),
    ('cruel',         'cruel',         'Cruel'),
    ('heist',         'heist',         'Heist'),
    ('robbery',       'robbery',       'Robbery'),
    ('ultra_clinical','ultra_clinical','Ultra Clinical'),
    ('dominant_win',  'dominant_win',  'Dominant Win'),
    ('gk_worldie',    'gk_worldie',    'GK Worldie'),
    ('gk_nightmare',  'gk_nightmare',  'GK Nightmare'),
    ('heroic_defence','heroic_defence','Heroic Defence'),
    ('fortress',      'fortress',      'Fortress'),
    ('home_bottled',  'home_bottled',  'Home Bottled'),
    ('clinical',      'clinical',      'Clinical'),
    ('wasteful',      'wasteful',      'Wasteful'),
]):
    clinical_df.loc[clinical_df[col] == val, 'game_tag'] = tag

In [13]:
clinical_df

,h_a,xG,xGA,npxG,npxGA,ppda,ppda_allowed,deep,deep_allowed,scored,...,defensive_collapse,gk_worldie,gk_nightmare,away_win,away_heist,home_bottled,perfect_heist,cruel,fortress,game_tag
0,h,4.62088,0.149750,3.86311,0.149750,"{'att': 203, 'def': 32}","{'att': 380, 'def': 15}",24,3,8,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,not_away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Ultra Clinical
1,h,3.99311,1.245090,3.23533,1.245090,"{'att': 157, 'def': 38}","{'att': 280, 'def': 23}",11,1,4,...,defensive_collapse,not_gk_worldie,gk_nightmare,not_away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Dominant Win
2,h,3.80823,0.429229,3.80823,0.429229,"{'att': 220, 'def': 22}","{'att': 372, 'def': 22}",12,1,5,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,not_away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Dominant Win
3,h,1.67360,1.631040,1.67360,1.631040,"{'att': 129, 'def': 33}","{'att': 266, 'def': 16}",18,4,1,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,not_away_win,not_away_heist,home_bottled,not_perfect_heist,not_cruel,not_fortress,Home Bottled
4,h,1.00047,1.564920,1.00047,1.564920,"{'att': 237, 'def': 19}","{'att': 345, 'def': 15}",6,1,3,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,not_away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Ultra Clinical
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17959,a,1.35523,1.728030,1.35523,1.728030,"{'att': 216, 'def': 26}","{'att': 242, 'def': 26}",15,5,1,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,not_away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Wasteful
17960,a,2.27029,1.059710,2.27029,1.059710,"{'att': 298, 'def': 26}","{'att': 371, 'def': 22}",7,1,3,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Clinical
17961,a,1.65387,0.666737,1.65387,0.666737,"{'att': 132, 'def': 25}","{'att': 186, 'def': 20}",4,1,3,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Clinical
17962,a,1.32818,2.745250,1.32818,2.745250,"{'att': 193, 'def': 21}","{'att': 306, 'def': 27}",10,9,1,...,not_defensive_collapse,gk_worldie,not_gk_nightmare,away_win,away_heist,not_home_bottled,perfect_heist,not_cruel,not_fortress,Perfect Heist


In [14]:
print(clinical_df.columns.tolist())
print(clinical_df['match_id'].isna().sum())

['h_a', 'xG', 'xGA', 'npxG', 'npxGA', 'ppda', 'ppda_allowed', 'deep', 'deep_allowed', 'scored', 'missed', 'xpts', 'result', 'date', 'wins', 'draws', 'loses', 'pts', 'npxGD', 'team_id', 'team_name', 'league', 'year', 'datetime', 'match_id', 'home_team', 'away_team', 'match_xGD', 'match_npxGD', 'xGA_diff', 'dominance', 'clinical', 'ultra_clinical', 'wasteful', 'ultra_wasteful', 'expected', 'heist', 'grand_heist', 'robbery', 'grand_robbery', 'dropped_points', 'stolen_point', 'dominant_win', 'match_total_goals', 'chaos', 'ghost', 'keepers_day', 'paradise_day', 'xGD_magnitude', 'dominance_gap', 'goals_vs_xG_ratio', 'goals_vs_xG_ratio_team', 'gk_save_ratio', 'xG_total', 'heroic_defence', 'routine_clean_sheet', 'defensive_collapse', 'gk_worldie', 'gk_nightmare', 'away_win', 'away_heist', 'home_bottled', 'perfect_heist', 'cruel', 'fortress', 'game_tag']
0


In [15]:
agg_dict = {
    'match_id':               'count',
    'scored':                 'sum',
    'missed':                 'sum',
    'xG':                     'sum',
    'xGA':                    'sum',
    'match_xGD':              'sum',
    'match_npxGD':            'sum',
    'dominance':              'sum',
    'xGD_magnitude':          'mean',
    'dominance_gap':          'mean',
    'gk_save_ratio':          'mean',
    'goals_vs_xG_ratio_team': 'mean',
    'clinical':           lambda x: (x == 'clinical').sum(),
    'ultra_clinical':     lambda x: (x == 'ultra_clinical').sum(),
    'wasteful':           lambda x: (x == 'wasteful').sum(),
    'ultra_wasteful':     lambda x: (x == 'ultra_wasteful').sum(),
    'heist':              lambda x: (x == 'heist').sum(),
    'grand_heist':        lambda x: (x == 'grand_heist').sum(),
    'robbery':            lambda x: (x == 'robbery').sum(),
    'grand_robbery':      lambda x: (x == 'grand_robbery').sum(),
    'dominant_win':       lambda x: (x == 'dominant_win').sum(),
    'dropped_points':     lambda x: (x == 'dropped_points').sum(),
    'stolen_point':       lambda x: (x == 'stolen_point').sum(),
    'heroic_defence':     lambda x: (x == 'heroic_defence').sum(),
    'gk_worldie':         lambda x: (x == 'gk_worldie').sum(),
    'gk_nightmare':       lambda x: (x == 'gk_nightmare').sum(),
    'home_bottled':       lambda x: (x == 'home_bottled').sum(),
    'away_heist':         lambda x: (x == 'away_heist').sum(),
    'fortress':           lambda x: (x == 'fortress').sum(),
    'perfect_heist':      lambda x: (x == 'perfect_heist').sum(),
    'cruel':              lambda x: (x == 'cruel').sum(),
}

team_season_ha = (
    clinical_df
    .groupby(['team_name', 'league', 'year', 'h_a'])
    .agg(agg_dict)
    .rename(columns={'match_id': 'games_played'})
    .reset_index()
)

# rate columns
team_season_ha['clinical_rate']     = (team_season_ha['clinical']     / team_season_ha['games_played']).round(3)
team_season_ha['heist_rate']        = (team_season_ha['heist']        / team_season_ha['games_played']).round(3)
team_season_ha['robbery_rate']      = (team_season_ha['robbery']      / team_season_ha['games_played']).round(3)
team_season_ha['wasteful_rate']     = (team_season_ha['wasteful']     / team_season_ha['games_played']).round(3)
team_season_ha['xG_per_game']       = (team_season_ha['xG']           / team_season_ha['games_played']).round(3)
team_season_ha['xGA_per_game']      = (team_season_ha['xGA']          / team_season_ha['games_played']).round(3)
team_season_ha['goals_per_game']    = (team_season_ha['scored']       / team_season_ha['games_played']).round(3)
team_season_ha['conceded_per_game'] = (team_season_ha['missed']       / team_season_ha['games_played']).round(3)
team_season_ha['xGD_per_game']      = (team_season_ha['match_xGD']    / team_season_ha['games_played']).round(3)

print(team_season_ha.shape)
team_season_ha.head()

(972, 44)


,team_name,league,year,h_a,games_played,scored,missed,xG,xGA,match_xGD,...,cruel,clinical_rate,heist_rate,robbery_rate,wasteful_rate,xG_per_game,xGA_per_game,goals_per_game,conceded_per_game,xGD_per_game
0,AC Milan,serie_a,2020,a,19,43,17,37.119001,24.307405,5.880999,...,0,0.526,0.263,0.000,0.474,1.954,1.279,2.263,0.895,0.310
1,AC Milan,serie_a,2020,h,19,31,24,37.928325,24.530668,-6.928325,...,0,0.421,0.053,0.053,0.579,1.996,1.291,1.632,1.263,-0.365
2,AC Milan,serie_a,2021,a,19,41,19,34.901264,19.913815,6.098736,...,0,0.526,0.105,0.053,0.474,1.837,1.048,2.158,1.000,0.321
3,AC Milan,serie_a,2021,h,19,28,12,32.436844,14.717525,-4.436844,...,0,0.368,0.053,0.053,0.579,1.707,0.775,1.474,0.632,-0.234
4,AC Milan,serie_a,2022,a,19,23,23,29.742495,22.612874,-6.742495,...,0,0.368,0.053,0.053,0.579,1.565,1.190,1.211,1.211,-0.355


In [16]:
team_season = (
    clinical_df
    .groupby(['team_name', 'league', 'year'])
    .agg({
        'match_id':           'count',
        'scored':             'sum',
        'missed':             'sum',
        'xG':                 'sum',
        'xGA':                'sum',
        'match_xGD':          'sum',
        'match_npxGD':        'sum',
        'dominance':          'sum',
        'xGD_magnitude':      'mean',
        'dominance_gap':      'mean',
        'gk_save_ratio':      'mean',
        'goals_vs_xG_ratio_team': 'mean',
        'clinical':           lambda x: (x == 'clinical').sum(),
        'ultra_clinical':     lambda x: (x == 'ultra_clinical').sum(),
        'wasteful':           lambda x: (x == 'wasteful').sum(),
        'ultra_wasteful':     lambda x: (x == 'ultra_wasteful').sum(),
        'heist':              lambda x: (x == 'heist').sum(),
        'grand_heist':        lambda x: (x == 'grand_heist').sum(),
        'robbery':            lambda x: (x == 'robbery').sum(),
        'grand_robbery':      lambda x: (x == 'grand_robbery').sum(),
        'dominant_win':       lambda x: (x == 'dominant_win').sum(),
        'dropped_points':     lambda x: (x == 'dropped_points').sum(),
        'stolen_point':       lambda x: (x == 'stolen_point').sum(),
        'heroic_defence':     lambda x: (x == 'heroic_defence').sum(),
        'gk_worldie':         lambda x: (x == 'gk_worldie').sum(),
        'gk_nightmare':       lambda x: (x == 'gk_nightmare').sum(),
        'home_bottled':       lambda x: (x == 'home_bottled').sum(),
        'away_heist':         lambda x: (x == 'away_heist').sum(),
        'fortress':           lambda x: (x == 'fortress').sum(),
        'perfect_heist':      lambda x: (x == 'perfect_heist').sum(),
        'cruel':              lambda x: (x == 'cruel').sum(),
    })
    .rename(columns={'match_id': 'games_played'})
    .reset_index()
)

team_season['xGD_attack']  = (team_season['scored'] - team_season['xG']).round(3)
team_season['xGD_defence'] = (team_season['missed'] - team_season['xGA']).round(3)

# re-derive rates
team_season['clinical_rate']     = (team_season['clinical']  / team_season['games_played']).round(3)
team_season['heist_rate']        = (team_season['heist']     / team_season['games_played']).round(3)
team_season['robbery_rate']      = (team_season['robbery']   / team_season['games_played']).round(3)
team_season['wasteful_rate']     = (team_season['wasteful']  / team_season['games_played']).round(3)
team_season['xG_per_game']       = (team_season['xG']        / team_season['games_played']).round(3)
team_season['xGA_per_game']      = (team_season['xGA']       / team_season['games_played']).round(3)
team_season['goals_per_game']    = (team_season['scored']    / team_season['games_played']).round(3)
team_season['conceded_per_game'] = (team_season['missed']    / team_season['games_played']).round(3)
team_season['xGD_per_game']      = (team_season['match_xGD'] / team_season['games_played']).round(3)

# sanity check
print(team_season[team_season['team_name'] == 'Manchester City'][['year','scored','xG','xGD_attack']])

     year  scored         xG  xGD_attack
272  2020      83  77.715218       5.285
273  2021      99  93.399873       5.600
274  2022      94  84.321567       9.678
275  2023      96  89.548753       6.451
276  2024      72  73.148430      -1.148


In [17]:
consistency = (
    team_season
    .groupby(['team_name', 'league'])
    .agg(
        seasons_played        = ('year', 'count'),
        seasons_clinical      = ('xGD_attack', lambda x: (x > 0).sum()),
        seasons_solid_defence = ('xGD_defence', lambda x: (x < 0).sum()),
        avg_xGD_attack        = ('xGD_attack', 'mean'),
        avg_xGD_defence       = ('xGD_defence', 'mean'),
        avg_clinical_rate     = ('clinical_rate', 'mean'),
        avg_heist_rate        = ('heist_rate', 'mean'),
        avg_robbery_rate      = ('robbery_rate', 'mean'),
        total_heists          = ('heist', 'sum'),
        total_grand_heists    = ('grand_heist', 'sum'),
        total_robberies       = ('robbery', 'sum'),
        total_gk_worldies     = ('gk_worldie', 'sum'),
        best_xGD_attack_season= ('xGD_attack', 'max'),
        worst_xGD_attack_season=('xGD_attack', 'min'),
    )
    .reset_index()
)

consistency['attack_verdict'] = pd.cut(
    consistency['seasons_clinical'],
    bins=[-1, 1, 3, 5],
    labels=['systematic_underperformer', 'inconsistent', 'systematic_overperformer']
)

consistency['defence_verdict'] = pd.cut(
    consistency['seasons_solid_defence'],
    bins=[-1, 1, 3, 5],
    labels=['systematic_defensive_underperformer', 'inconsistent_defence', 'systematic_defensive_overperformer']
)

print(consistency.shape)
consistency.sort_values('avg_xGD_attack', ascending=False).head(10)

(132, 18)


,team_name,league,seasons_played,seasons_clinical,seasons_solid_defence,avg_xGD_attack,avg_xGD_defence,avg_clinical_rate,avg_heist_rate,avg_robbery_rate,total_heists,total_grand_heists,total_robberies,total_gk_worldies,best_xGD_attack_season,worst_xGD_attack_season,attack_verdict,defence_verdict
14,Bayer Leverkusen,bundesliga,5,5,2,7.7354,-0.8132,0.5412,0.0764,0.0588,13,3,10,5,13.688,3.316,systematic_overperformer,inconsistent_defence
15,Bayern Munich,bundesliga,5,3,1,6.1252,3.6596,0.5294,0.0234,0.0704,4,1,12,2,23.068,-2.906,inconsistent,systematic_defensive_underperformer
20,Borussia Dortmund,bundesliga,5,4,1,6.1056,0.4128,0.5824,0.0824,0.0824,14,1,14,6,19.354,-3.164,systematic_overperformer,systematic_defensive_underperformer
57,Holstein Kiel,bundesliga,1,1,0,5.4320,9.4130,0.4710,0.0590,0.0590,2,1,2,1,5.432,5.432,systematic_underperformer,systematic_defensive_underperformer
34,Crotone,serie_a,1,1,0,5.3600,14.5170,0.4470,0.0530,0.0530,2,0,2,0,5.360,5.360,systematic_underperformer,systematic_defensive_underperformer
77,Manchester City,premier_league,5,4,3,5.1732,-2.2414,0.5106,0.0476,0.0422,9,3,8,6,9.678,-1.148,systematic_overperformer,inconsistent_defence
116,Tottenham,premier_league,5,4,4,4.7648,-2.0416,0.4524,0.0738,0.0790,14,1,15,10,12.166,-1.363,systematic_overperformer,systematic_defensive_overperformer
67,Leicester,premier_league,4,3,2,4.6665,-0.4620,0.5000,0.1120,0.0790,17,3,12,11,12.511,-5.058,inconsistent,inconsistent_defence
62,Lazio,serie_a,5,3,2,3.7996,0.1326,0.4632,0.0896,0.0792,17,3,15,7,17.709,-4.319,inconsistent,inconsistent_defence
101,Rennes,ligue_1,5,3,3,3.5936,-0.5900,0.4800,0.0598,0.1108,11,2,20,6,13.759,-3.003,inconsistent,inconsistent_defence


In [18]:
team_season.to_csv('../data/processed/team_season.csv', index=False)
team_season_ha.to_csv('../data/processed/team_season_ha.csv', index=False)
consistency.to_csv('../data/processed/consistency.csv', index=False)
clinical_df.to_csv('../data/processed/clinical_games.csv', index=False)

In [ ]:
from pathlib import Path

LEAGUE_SIZES = {
    'premier_league': 20, 'la_liga': 20, 'serie_a': 20,
    'bundesliga': 18, 'ligue_1': 20
}

def add_zscore(df, group_cols, value_col):
    col = f'{value_col}_zscore'
    df[col] = df.groupby(group_cols)[value_col].transform(
        lambda x: (x - x.mean()) / max(x.std(), 1e-9)
    )
    return df

for col in ['xGD_attack', 'xGD_defence', 'heist_rate', 'clinical_rate', 'robbery_rate']:
    team_season = add_zscore(team_season, ['league', 'year'], col)

team_season['xGD_attack_rank'] = team_season.groupby(['league','year'])['xGD_attack'].rank(ascending=False).astype(int)
team_season['heist_rate_rank']  = team_season.groupby(['league','year'])['heist_rate'].rank(ascending=False).astype(int)

consistency['attack_verdict'] = consistency.apply(
    lambda r: 'systematic_overperformer'  if r['seasons_clinical']      / r['seasons_played'] >= 0.8
         else 'systematic_underperformer' if r['seasons_clinical']      / r['seasons_played'] <= 0.2
         else 'inconsistent', axis=1
)
consistency['defence_verdict'] = consistency.apply(
    lambda r: 'systematic_defensive_overperformer'  if r['seasons_solid_defence'] / r['seasons_played'] >= 0.8
         else 'systematic_defensive_underperformer' if r['seasons_solid_defence'] / r['seasons_played'] <= 0.2
         else 'inconsistent_defence', axis=1
)

rag_dir = Path('../data/rag_findings')
rag_dir.mkdir(exist_ok=True)

for league in consistency['league'].unique():
    league_teams       = consistency[consistency['league'] == league]
    league_teams_multi = league_teams[league_teams['seasons_played'] > 1]  # fix: seasons_played not season_played
    league_season      = team_season[team_season['league'] == league]
    league_size        = LEAGUE_SIZES.get(league, 20)

    top_clinical   = league_teams_multi.nlargest(1,  'avg_xGD_attack').iloc[0]
    most_wasteful  = league_teams_multi.nsmallest(1, 'avg_xGD_attack').iloc[0]
    best_defence   = league_teams_multi.nsmallest(1, 'avg_xGD_defence').iloc[0]
    most_heists    = league_teams_multi.nlargest(1,  'total_heists').iloc[0]
    most_robberies = league_teams_multi.nlargest(1,  'total_robberies').iloc[0]
    sys_over       = (league_teams_multi['attack_verdict'] == 'systematic_overperformer').sum()
    sys_under      = (league_teams_multi['attack_verdict'] == 'systematic_underperformer').sum()

    league_header = f"""
=====================================
XG ANALYSIS | LEAGUE: {league.upper()} | SEASONS: 2020-2024
=====================================
LEAGUE SUMMARY (multi-season teams only):
- Most clinical team (avg xGD attack): {top_clinical['team_name']} ({top_clinical['avg_xGD_attack']:+.2f} goals/season vs xG)
- Most wasteful team: {most_wasteful['team_name']} ({most_wasteful['avg_xGD_attack']:+.2f} goals/season vs xG)
- Best defence vs xG: {best_defence['team_name']} ({best_defence['avg_xGD_defence']:+.2f} goals conceded vs xGA)
- Most heists: {most_heists['team_name']} ({int(most_heists['total_heists'])} across tracked seasons)
- Most robberies: {most_robberies['team_name']} ({int(most_robberies['total_robberies'])} across tracked seasons)
- Systematic overperformers: {sys_over} | Systematic underperformers: {sys_under}
-------------------------------------
TEAM PROFILES:
"""

    blocks = []
    for _, row in league_teams.iterrows():
        team_seasons = league_season[league_season['team_name'] == row['team_name']].sort_values('year')
        if team_seasons.empty:
            continue

        best_idx   = team_seasons['xGD_attack'].idxmax()
        worst_idx  = team_seasons['xGD_attack'].idxmin()
        best_year  = team_seasons.loc[best_idx,  'year']
        worst_year = team_seasons.loc[worst_idx, 'year']
        best_val   = team_seasons.loc[best_idx,  'xGD_attack']
        worst_val  = team_seasons.loc[worst_idx, 'xGD_attack']

        progression = " → ".join(
            f"{int(r['year'])}: {r['xGD_attack']:+.1f}".replace("-0.0", "0.0")
            for _, r in team_seasons.iterrows()
        )

        n = len(team_seasons)
        if n >= 3:
            half        = n // 2
            first_half  = team_seasons.head(half)['xGD_attack'].mean()
            second_half = team_seasons.tail(half)['xGD_attack'].mean()
            trend       = 'improving' if second_half > first_half else 'declining'
        else:
            trend = 'insufficient_data'

        if row['avg_heist_rate'] > 0.12:
            style = 'counter-attacking/heist specialists'
        elif row['avg_clinical_rate'] > 0.50 and row['attack_verdict'] == 'systematic_overperformer':
            style = 'consistently clinical finishers'
        elif row['avg_robbery_rate'] > 0.10:
            style = 'chance-dominant but wasteful'
        elif row['avg_xGD_defence'] < -3 and row['defence_verdict'] in ['systematic_defensive_overperformer', 'inconsistent_defence']:
            style = 'defensively elite'
        else:
            style = 'balanced'

        avg_xGD_zscore     = team_seasons['xGD_attack_zscore'].mean()
        avg_heist_zscore   = team_seasons['heist_rate_zscore'].mean()
        avg_defence_zscore = team_seasons['xGD_defence_zscore'].mean()
        avg_xGD_rank       = team_seasons['xGD_attack_rank'].mean()
        avg_heist_rank     = team_seasons['heist_rate_rank'].mean()

        if n < 3:
            trend_note = f"only {n} season(s) tracked — insufficient for trend"
        elif trend == 'declining':
            trend_note = f"declining trend since {best_year} peak"
        else:
            trend_note = f"recovering since {worst_year} trough"

        # single season caveat
        sample_flag    = 'single_season' if n == 1 else 'multi_season'
        seasons_caveat = " Note: single-season sample — treat with caution." if n == 1 else ""

        insight = (
            f"{row['team_name']} are a {style} side in {league.replace('_',' ').title()} — "
            f"averaging {row['avg_xGD_attack']:+.2f} xGD attack ({avg_xGD_zscore:+.1f}σ vs league), "
            f"solid in defence {row['seasons_solid_defence']}/{row['seasons_played']} seasons. "
            f"{trend_note.capitalize()}.{seasons_caveat}"
        )

        block = f"""
TEAM: {row['team_name']} | LEAGUE: {league} | SEASONS TRACKED: {row['seasons_played']} | SAMPLE: {sample_flag}
FINISHING IDENTITY: {style} | TREND: {trend}
ATTACK: avg_xGD_attack={row['avg_xGD_attack']:+.2f} | clinical_rate={row['avg_clinical_rate']:.1%} | seasons_overperforming={row['seasons_clinical']}/{row['seasons_played']}
DEFENCE: avg_xGD_defence={row['avg_xGD_defence']:+.2f} | seasons_solid={row['seasons_solid_defence']}/{row['seasons_played']}
LEAGUE RANK: avg_xGD_attack_rank={avg_xGD_rank:.1f} of {league_size} | avg_heist_rank={avg_heist_rank:.1f} of {league_size}
Z-SCORES: xGD_attack={avg_xGD_zscore:+.2f}σ | heist_rate={avg_heist_zscore:+.2f}σ | xGD_defence={avg_defence_zscore:+.2f}σ
SEASON_PROGRESSION (xGD_attack): {progression}
PEAK: best season {best_year} ({best_val:+.2f} xGD attack) | worst season {worst_year} ({worst_val:+.2f} xGD attack)
GAME TAGS: heists={int(row['total_heists'])} (rate={row['avg_heist_rate']:.1%}) | grand_heists={int(row['total_grand_heists'])} | robberies={int(row['total_robberies'])} | gk_worldies={int(row['total_gk_worldies'])}
VERDICT: attack={row['attack_verdict']} | defence={row['defence_verdict']}
INSIGHT: {insight}
---"""
        blocks.append(block)

    output = league_header + "\n".join(blocks)
    (rag_dir / f"xg_analysis_{league}.txt").write_text(output)
    print(f"Written: xg_analysis_{league}.txt")

Written: xg_analysis_serie_a.txt
Written: xg_analysis_ligue_1.txt
Written: xg_analysis_la_liga.txt
Written: xg_analysis_bundesliga.txt
Written: xg_analysis_premier_league.txt
